In [ ]:
#!/usr/bin/env python3
"""
Calculate nucleic-acid heavy-atom RMSD for multiple systems and replicas.

Each replica is aligned to its own first trajectory frame.

Example:
    python calculate_rmsd_triplicates.py \
        --manifest configs/systems/rmsd_na_parallel.local.csv \
        --output data/processed/rmsd/rmsd_na_parallel.npz
"""

import argparse
import csv
from collections import OrderedDict
from pathlib import Path

import mdtraj as md
import numpy as np


def parse_arguments():
    """讀取命令列參數。"""

    parser = argparse.ArgumentParser(
        description="Calculate RMSD means and SDs across MD replicas."
    )

    parser.add_argument(
        "--manifest",
        required=True,
        type=Path,
        help="CSV file describing systems, topologies, and trajectories.",
    )

    parser.add_argument(
        "--output",
        required=True,
        type=Path,
        help="Output NPZ file.",
    )

    parser.add_argument(
        "--chunk-size",
        type=int,
        default=1000,
        help="Number of frames loaded per chunk. Default: 1000.",
    )

    return parser.parse_args()


def load_manifest(manifest_path):
    """讀取並按照 system_id 整理 CSV 系統資訊。"""

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"找不到 manifest：{manifest_path}"
        )

    required_columns = {
        "system_id",
        "label",
        "color",
        "topology",
        "replica",
        "trajectory",
    }

    systems = OrderedDict()

    with manifest_path.open(
        mode="r",
        encoding="utf-8-sig",
        newline="",
    ) as handle:
        reader = csv.DictReader(handle)

        if reader.fieldnames is None:
            raise ValueError("Manifest 沒有標題列。")

        missing_columns = required_columns - set(reader.fieldnames)

        if missing_columns:
            raise ValueError(
                "Manifest 缺少欄位："
                + ", ".join(sorted(missing_columns))
            )

        for row in reader:
            system_id = row["system_id"].strip()

            if not system_id:
                continue

            systems.setdefault(
                system_id,
                {
                    "label": row["label"].strip(),
                    "color": row["color"].strip(),
                    "topology": Path(row["topology"].strip()),
                    "replicas": [],
                },
            )

            systems[system_id]["replicas"].append(
                {
                    "replica": int(row["replica"]),
                    "trajectory": Path(
                        row["trajectory"].strip()
                    ),
                }
            )

    return systems


def calculate_replica_rmsd(
    topology_path,
    trajectory_path,
    chunk_size,
):
    """
    計算一個 replica 的核酸重原子 RMSD。

    使用 iterload 分批讀取軌跡，避免一次將完整 DCD 載入記憶體。
    RMSD reference 為該 replica 的第一個 trajectory frame。
    """

    if not topology_path.exists():
        raise FileNotFoundError(
            f"找不到 topology：{topology_path}"
        )

    if not trajectory_path.exists():
        raise FileNotFoundError(
            f"找不到 trajectory：{trajectory_path}"
        )

    topology = md.load_topology(str(topology_path))

    atom_indices = topology.select(
        "nucleic and not element H"
    )

    if len(atom_indices) == 0:
        raise ValueError(
            "Topology 中找不到 nucleic heavy atoms。"
        )

    rmsd_chunks = []
    reference = None

    trajectory_iterator = md.iterload(
        str(trajectory_path),
        top=str(topology_path),
        atom_indices=atom_indices,
        chunk=chunk_size,
    )

    for chunk in trajectory_iterator:
        if reference is None:
            reference = chunk[0]

        # md.rmsd 本身會執行最佳化疊合，不需要先呼叫 superpose
        chunk_rmsd = md.rmsd(
            chunk,
            reference,
            frame=0,
        )

        # MDTraj 輸出為 nm，轉換為 Å
        rmsd_chunks.append(
            chunk_rmsd * 10.0
        )

    if not rmsd_chunks:
        raise ValueError("Trajectory 沒有可讀取的 frame。")

    return np.concatenate(rmsd_chunks)


def calculate_system_statistics(
    system_id,
    system_info,
    chunk_size,
):
    """計算一個系統所有 replicas 的 RMSD 統計。"""

    topology_path = system_info["topology"]
    replica_results = []

    sorted_replicas = sorted(
        system_info["replicas"],
        key=lambda item: item["replica"],
    )

    print(f"\nSystem: {system_id}")
    print(f"Label: {system_info['label']}")

    for replica_info in sorted_replicas:
        replica_id = replica_info["replica"]
        trajectory_path = replica_info["trajectory"]

        print(
            f"  Replica {replica_id}: "
            f"{trajectory_path.name}"
        )

        try:
            rmsd_values = calculate_replica_rmsd(
                topology_path=topology_path,
                trajectory_path=trajectory_path,
                chunk_size=chunk_size,
            )

        except (OSError, ValueError) as error:
            print(f"  [警告] Replica {replica_id} 失敗：{error}")
            continue

        replica_results.append(rmsd_values)

        print(
            f"  [OK] {len(rmsd_values)} frames, "
            f"mean={np.mean(rmsd_values):.3f} Å"
        )

    if not replica_results:
        print(f"  [跳過] {system_id} 沒有有效 replicas。")
        return None

    minimum_frames = min(
        len(values)
        for values in replica_results
    )

    trimmed_results = [
        values[:minimum_frames]
        for values in replica_results
    ]

    stacked_rmsd = np.vstack(trimmed_results)

    replica_count = stacked_rmsd.shape[0]

    mean_rmsd = np.mean(
        stacked_rmsd,
        axis=0,
    )

    # 三重複使用 sample standard deviation
    std_rmsd = np.std(
        stacked_rmsd,
        axis=0,
        ddof=1 if replica_count > 1 else 0,
    )

    print(
        f"  完成：replicas={replica_count}, "
        f"frames={minimum_frames}"
    )

    return {
        "mean": mean_rmsd,
        "std": std_rmsd,
        "frames": minimum_frames,
        "replicas": replica_count,
    }


def main():
    """主程式。"""

    args = parse_arguments()

    systems = load_manifest(
        args.manifest
    )

    if not systems:
        raise RuntimeError(
            "Manifest 中沒有有效系統。"
        )

    final_rmsd_data = {}
    system_meta = {}

    print("Starting RMSD calculation")

    for system_id, system_info in systems.items():
        result = calculate_system_statistics(
            system_id=system_id,
            system_info=system_info,
            chunk_size=args.chunk_size,
        )

        if result is None:
            continue

        label = system_info["label"]

        final_rmsd_data[label] = result
        system_meta[label] = {
            "color": system_info["color"],
            "system_id": system_id,
        }

    if not final_rmsd_data:
        raise RuntimeError(
            "沒有任何有效 RMSD 結果可以儲存。"
        )

    args.output.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    # 保持與既有 plot_rmsd.py 相容
    np.savez_compressed(
        args.output,
        data=final_rmsd_data,
        meta=system_meta,
    )

    print(f"\nRMSD data saved to: {args.output}")


if __name__ == "__main__":
    main()